# 00 Load the world (optional helper)

This notebook puts the Strandvik world into your lab:

1. **Documents**: uploads `data/documents/*.md` to the `documents` container of your storage account (for Foundry IQ).
2. **Lakehouse**: creates the lakehouse `strandvik_lakehouse` in your Fabric workspace and loads every `data/lakehouse/*.csv` as a table (for Fabric IQ).

**You do not need code for this.** Business participants can do exactly the same by hand:

- Azure portal > your resource group > the storage account > Storage browser > Blob containers > `documents` > Upload.
- Fabric portal > your workspace > New item > Lakehouse > Get data > Upload files, then on each CSV: ... > Load to Tables.

Before you start: run `bash create-env.sh` in the hack folder so `.env` is filled in.

In [ ]:
# Run once if the packages are missing (the Codespace installs them for you).
# Packages come through the Microsoft package proxy.
# %pip install -q -r requirements.txt --index-url https://packagefeedproxy.microsoft.io/pypi/simple/

In [ ]:
# Settings come from ../.env, written by create-env.sh (or create-env.ps1).
# You sign in once with "az login"; the notebook reuses that sign-in.
import os, json, time
from pathlib import Path
import requests
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, ChainedTokenCredential, DefaultAzureCredential

HACK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(HACK_DIR / ".env", override=True)
env = os.environ.get

credential = ChainedTokenCredential(AzureCliCredential(process_timeout=30),
                                    DefaultAzureCredential(exclude_cli_credential=True))

def call(method, url, scope, body=None, ok=(200, 201, 202, 204), headers=None, data=None):
    """One REST call with an Entra token. Retries throttling. Raises on any other error."""
    hdrs = {"Authorization": "Bearer " + credential.get_token(scope).token}
    if body is not None:
        hdrs["Content-Type"] = "application/json"
    hdrs.update(headers or {})
    for attempt in range(8):
        r = requests.request(method, url, headers=hdrs, json=body, data=data, timeout=180)
        # 430 = the Fabric capacity is busy (small F2 capacity), 429/5xx = throttling.
        if r.status_code in (429, 430, 500, 502, 503) and attempt < 7:
            time.sleep(int(r.headers.get("Retry-After", 10 + 10 * attempt)))
            continue
        break
    if r.status_code not in ok:
        raise RuntimeError(f"{method} {url.split('?')[0]} -> HTTP {r.status_code}: {r.text[:1000]}")
    return r

## 1. Upload the documents to Blob Storage

In [ ]:
STORAGE = env("STORAGE_ACCOUNT")
CONTAINER = env("DOCUMENTS_CONTAINER", "documents")
STORAGE_SCOPE = "https://storage.azure.com/.default"

docs = sorted((HACK_DIR / "data" / "documents").glob("*.md"))
for f in docs:
    # A block blob PUT with your Entra token (you hold Storage Blob Data Contributor).
    call("PUT", f"https://{STORAGE}.blob.core.windows.net/{CONTAINER}/{f.name}", STORAGE_SCOPE, ok=(201,),
         headers={"x-ms-blob-type": "BlockBlob", "x-ms-version": "2025-01-05",
                  "Content-Type": "text/markdown; charset=utf-8"},
         data=f.read_bytes())
    print("uploaded", f.name)
print(f"{len(docs)} documents in {STORAGE}/{CONTAINER}")

## 2. Load the lakehouse in Fabric

Needs `FABRIC_WORKSPACE_ID` in `.env` (your workspace on your F2 capacity). An F2 capacity runs few table loads at a time, so the notebook waits when Fabric answers HTTP 430 ("capacity busy").

In [ ]:
FABRIC = "https://api.fabric.microsoft.com/v1"
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"
WS = env("FABRIC_WORKSPACE_ID")
LAKEHOUSE = "strandvik_lakehouse"
assert WS, "FABRIC_WORKSPACE_ID is empty: create your Fabric workspace, then run create-env.sh again."

def fabric(method, path, body=None, ok=(200, 201, 202)):
    """Fabric REST call. Long-running operations (202 + Location) are awaited."""
    r = call(method, path if path.startswith("http") else FABRIC + path, FABRIC_SCOPE, body, ok)
    if r.status_code == 202 and r.headers.get("Location"):
        loc = r.headers["Location"]
        for _ in range(180):
            time.sleep(int(r.headers.get("Retry-After", 5)))
            s = call("GET", loc, FABRIC_SCOPE, ok=(200, 202))
            st = s.json() if s.content else {}
            if st.get("status") in ("Succeeded", "Completed") or (s.status_code == 200 and "status" not in st):
                res = call("GET", loc + "/result", FABRIC_SCOPE, ok=(200, 400, 404))
                return res.json() if res.status_code == 200 and res.content else st
            if st.get("status") in ("Failed", "Cancelled"):
                raise RuntimeError(f"Fabric operation failed: {st}")
        raise TimeoutError("Fabric operation did not finish in 15 minutes.")
    return r.json() if r.content else {}

def find(kind, name):
    return next((i for i in fabric("GET", f"/workspaces/{WS}/{kind}").get("value", []) if i["displayName"] == name), None)

lh = find("lakehouses", LAKEHOUSE) or fabric("POST", f"/workspaces/{WS}/lakehouses", {"displayName": LAKEHOUSE})
lh = lh if lh.get("id") else find("lakehouses", LAKEHOUSE)
LH = lh["id"]
print("lakehouse", LAKEHOUSE, LH)

In [ ]:
ONELAKE_SCOPE = "https://storage.azure.com/.default"

def onelake_upload(path, data):
    """Create, append and flush one file in the lakehouse Files area (OneLake DFS API)."""
    base = f"https://onelake.dfs.fabric.microsoft.com/{WS}/{LH}/{path}"
    call("PUT", base + "?resource=file&overwrite=true", ONELAKE_SCOPE, ok=(200, 201))
    call("PATCH", base + "?action=append&position=0", ONELAKE_SCOPE, ok=(200, 202),
         headers={"Content-Type": "application/octet-stream"}, data=data)
    call("PATCH", f"{base}?action=flush&position={len(data)}", ONELAKE_SCOPE, ok=(200,))

tables = []
for f in sorted((HACK_DIR / "data" / "lakehouse").glob("*.csv")):
    onelake_upload(f"Files/strandvik/{f.name}", f.read_bytes())
    body = {"relativePath": f"Files/strandvik/{f.name}", "pathType": "File", "mode": "Overwrite",
            "formatOptions": {"format": "Csv", "header": True, "delimiter": ","}}
    for attempt in range(12):
        try:
            fabric("POST", f"/workspaces/{WS}/lakehouses/{LH}/tables/{f.stem}/load", body)
            break
        except RuntimeError as e:
            # HTTP 430 TooManyRequestsForCapacity: the previous load is still running.
            if "TooManyRequestsForCapacity" not in str(e) and "430" not in str(e) or attempt == 11:
                raise
            print("  capacity busy, waiting before loading", f.stem)
            time.sleep(20)
    tables.append(f.stem)
    print("loaded table", f.stem)

In [ ]:
# New Delta tables reach the SQL analytics endpoint after a metadata sync. The Fabric data
# agent reads its schema from that endpoint, so sync now instead of waiting.
props = fabric("GET", f"/workspaces/{WS}/lakehouses/{LH}")["properties"]
sql = props.get("sqlEndpointProperties") or {}
if sql.get("id"):
    try:
        fabric("POST", f"/workspaces/{WS}/sqlEndpoints/{sql['id']}/refreshMetadata", {})
        print("SQL endpoint metadata refreshed")
    except RuntimeError as e:
        print("Refresh skipped, wait 2 minutes before creating the data agent:", str(e)[:150])

# Check: every table with its columns. (If you ever create a data agent by REST, its
# datasource must list these column elements, or it cannot see the columns.)
import pandas as pd
rows = [{"table": t, "columns": ", ".join(pd.read_csv(HACK_DIR / "data" / "lakehouse" / f"{t}.csv", nrows=1).columns)}
        for t in tables]
pd.DataFrame(rows)